# VitaVision Pre-Training Data Preparation

This notebook prepares the cleaned and labeled VitaVision dataset for machine learning training.

The focus here is **not model training yet**. This stage prepares a reliable modeling dataset using strong data science practices:

1. Load the final labeled dataset.
2. Standardize columns and values.
3. Define features, target, and patient groups.
4. Split data using patient-level splitting to reduce data leakage.
5. Compare label and nutrient distributions across train, validation, and test sets.
6. Prepare preprocessing objects for numeric and categorical features.
7. Calculate class weights to handle class imbalance.
8. Optionally save prepared splits for the training notebook.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42

## 2. Load Final Labeled Dataset

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("vitavision_final_labeled_dataset.csv"),
    Path("data/vitavision_final_labeled_dataset.csv"),
]

data_path = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("Could not find vitavision_final_labeled_dataset.csv. Run this notebook from the project root or data folder.")

df_raw = pd.read_csv(data_path)

print(f"Dataset path: {data_path.resolve()}")
print(f"Shape: {df_raw.shape}")

df_raw.head()

## 3. Standardize Dataset for Modeling

This step keeps the modeling table clean and predictable. It also standardizes nutrient naming so the training notebook and app can use one naming convention.

In [ ]:
required_columns = ["SEQN", "Age", "Gender", "Nutrient", "Value", "Label"]
missing_columns = [col for col in required_columns if col not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df_raw[required_columns].copy()

df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = pd.to_numeric(df["Gender"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["Nutrient"] = df["Nutrient"].astype(str).str.strip()
df["Label"] = df["Label"].astype(str).str.strip()

nutrient_name_map = {
    "Vitamin_D": "Vitamin D",
    "Vitamin_C": "Vitamin C",
    "Vitamin_A": "Vitamin A",
    "Vitamin_E": "Vitamin E",
    "Vitamin_K": "Vitamin K",
    "B12": "Vitamin B12",
    "B6": "Vitamin B6",
}

df["Nutrient"] = df["Nutrient"].replace(nutrient_name_map)

df = df.dropna(subset=required_columns).copy()
df["SEQN"] = df["SEQN"].astype(int)
df["Gender"] = df["Gender"].astype(int)

df.head()

## 4. Final Modeling Validation

These checks stop the workflow early if the modeling dataset has invalid values.

In [ ]:
expected_labels = ["Deficient", "Normal", "Excessive"]

validation_checks = {
    "no_missing_values": df[required_columns].isna().sum().sum() == 0,
    "valid_gender_values": df["Gender"].isin([1, 2]).all(),
    "valid_age_range": df["Age"].between(0, 120).all(),
    "positive_lab_values": (df["Value"] > 0).all(),
    "valid_labels_only": df["Label"].isin(expected_labels).all(),
}

validation_report = pd.DataFrame(
    [{"check": check, "passed": passed} for check, passed in validation_checks.items()]
)

validation_report

In [ ]:
if not all(validation_checks.values()):
    failed = [check for check, passed in validation_checks.items() if not passed]
    raise ValueError(f"Modeling validation failed: {failed}")

print("Modeling dataset passed validation checks.")

## 5. Define Features, Target, and Groups

`SEQN` is the patient identifier. We do not use it as a feature, but we use it as the group key for splitting.

Why this matters: one patient can have multiple nutrient rows. If the same patient appears in both training and testing, the evaluation may look better than it really is.

In [ ]:
FEATURES = ["Age", "Gender", "Nutrient", "Value"]
TARGET = "Label"
GROUP = "SEQN"

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()

print("Features:", FEATURES)
print("Target:", TARGET)
print("Group column:", GROUP)
print("X shape:", X.shape)
print("y shape:", y.shape)

## 6. Patient-Level Train/Test Split

We split by patient ID using `GroupShuffleSplit`. This prevents the same patient from appearing in both training and test sets.

In [ ]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_val_idx, test_idx = next(group_splitter.split(df, y, groups=groups))

train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train+Validation shape:", train_val_df.shape)
print("Test shape:", test_df.shape)

## 7. Patient-Level Validation Split

From the training portion, we create a validation set. Final ratio is approximately:

- 64% train
- 16% validation
- 20% test

In [ ]:
val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, val_idx = next(
    val_splitter.split(
        train_val_df,
        train_val_df[TARGET],
        groups=train_val_df[GROUP],
    )
)

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

split_sizes = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "patients": [train_df[GROUP].nunique(), val_df[GROUP].nunique(), test_df[GROUP].nunique()],
})
split_sizes["row_percent"] = (split_sizes["rows"] / len(df) * 100).round(2)

split_sizes

## 8. Leakage Check

This confirms that no patient ID appears in more than one split.

In [ ]:
train_patients = set(train_df[GROUP])
val_patients = set(val_df[GROUP])
test_patients = set(test_df[GROUP])

leakage_report = pd.DataFrame({
    "overlap": ["train_vs_validation", "train_vs_test", "validation_vs_test"],
    "overlapping_patients": [
        len(train_patients & val_patients),
        len(train_patients & test_patients),
        len(val_patients & test_patients),
    ],
})

leakage_report

## 9. Compare Label Distributions Across Splits

Group-based splitting is stronger against leakage, but it can slightly change class proportions. We check that here.

In [ ]:
def label_distribution(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = data[TARGET].value_counts().rename("count").to_frame()
    counts["percent"] = (counts["count"] / len(data) * 100).round(2)
    counts["split"] = split_name
    return counts.reset_index().rename(columns={"index": TARGET})

label_dist = pd.concat([
    label_distribution(train_df, "train"),
    label_distribution(val_df, "validation"),
    label_distribution(test_df, "test"),
], ignore_index=True)

label_dist.pivot(index=TARGET, columns="split", values="percent").fillna(0).round(2)

## 10. Compare Nutrient Coverage Across Splits

In [ ]:
def nutrient_distribution(data: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = data["Nutrient"].value_counts().rename("count").to_frame()
    counts["percent"] = (counts["count"] / len(data) * 100).round(2)
    counts["split"] = split_name
    return counts.reset_index().rename(columns={"index": "Nutrient"})

nutrient_dist = pd.concat([
    nutrient_distribution(train_df, "train"),
    nutrient_distribution(val_df, "validation"),
    nutrient_distribution(test_df, "test"),
], ignore_index=True)

nutrient_dist.pivot(index="Nutrient", columns="split", values="percent").fillna(0).round(2)

## 11. Build Preprocessing Object

This object will be used inside the training pipeline.

- Numeric features: `Age`, `Gender`, `Value`
- Categorical features: `Nutrient`

`OneHotEncoder` handles nutrient names. `StandardScaler` is useful for many models. Tree models such as Random Forest do not require scaling, but keeping this object makes the pipeline ready for comparing multiple algorithms later.

In [ ]:
numeric_features = ["Age", "Gender", "Value"]
categorical_features = ["Nutrient"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

preprocessor

## 12. Prepare X and y for Each Split

In [ ]:
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

## 13. Class Weights for Imbalanced Labels

The dataset contains more `Normal` cases than abnormal cases. Class weights help the model pay more attention to minority classes.

In [ ]:
classes = np.array(sorted(y_train.unique()))
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

class_weight = dict(zip(classes, weights))

pd.DataFrame({
    "class": list(class_weight.keys()),
    "weight": list(class_weight.values()),
}).round(4)

## 14. Optional: Save Prepared Splits

By default, saving is disabled. Change `SAVE_SPLITS` to `True` if you want to create CSV files for the training notebook.

In [ ]:
SAVE_SPLITS = False
OUTPUT_DIR = Path("prepared_splits")

if SAVE_SPLITS:
    OUTPUT_DIR.mkdir(exist_ok=True)
    train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
    val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)
    test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)
    print(f"Saved prepared splits to: {OUTPUT_DIR.resolve()}")
else:
    print("Saving disabled. Set SAVE_SPLITS = True to export split files.")

## 15. Pre-Training Summary

At this point, the dataset is ready for model training.

Recommended next training setup:

- Use a pipeline: `preprocessor + classifier`.
- Start with `RandomForestClassifier(class_weight="balanced")`.
- Evaluate with Accuracy, Macro F1-score, classification report, and confusion matrix.
- Use the test set only once for final evaluation.
- In the report, mention that patient-level splitting was used to reduce leakage risk.